In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path


# Find the project root from the current directory
def find_project_root(start_path: Path) -> Path:
    """Find project root by locating the src/ directory and requirements.txt."""
    for path in [start_path, *start_path.parents]:
        if (path / "src").is_dir() and (path / "requirements.txt").is_file():
            return path

    raise FileNotFoundError(
        "Could not locate project root containing both 'src/' and 'requirements.txt'."
    )


PROJECT_ROOT = find_project_root(Path.cwd())


# Ensure Python recognizes the 'src' package
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# Import project paths
from src.paths import RAW_DIR, INTERIM_DIR


# Import pipeline modules
from src.pipeline.api import merge_data
from src.pipeline.merging.planner import MERGE_TABLE_SPECS
from src.pipeline.run_merge import save_merged_dataset


# Check the environment
print(f"Project root: {PROJECT_ROOT}")
print(f"Data raw path: {RAW_DIR}")
print(f"Interim data path: {INTERIM_DIR}")

In [ ]:
# Merge and validate the full registered dataset.
merge_result = merge_data(tables=list(MERGE_TABLE_SPECS))
df_merged = merge_result.data

# Save to interim, trying Parquet first with CSV fallback.
saved_path = save_merged_dataset(df_merged, output_dir=INTERIM_DIR)

# Quick check of dimensions and data
print(f"Final shape: {df_merged.shape}")
df_merged.head(3)

In [ ]:
# Verify exactly 25 columns and the correct data types (datetime, string for zip code)
df_merged.info(verbose=True, show_counts=True)